[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/eddy_brake/eddy_brake.ipynb)

# Eddy-current braking through directional conductors

A conducting disc turns beneath a magnetic patch. Motion drives currents, and their magnetic force brakes the disc. Conductive fibres guide those currents: a local conductivity extensor maps electric field to current, while edge differences, reconstruction and their adjoints assemble the charge-balance equation. We compare isotropic, radial-fibre and circumferential-fibre discs with the same mean conductivity.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.mesh import Mesh, as_scalar
from examples.electromagnetism.eddy_brake import core, render

In [ ]:
# The shared mesh geometry and its fields use the same Euclidean algebra.
ga = core.ga
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Conductivity = ga.gatype((Vector, Vector))                      # Vector <- Vector
Rotor = ga.gatype.rotor()

In [ ]:
radius = 0.1                                       # m
thickness = 0.001                                  # m
reference_conductivity = 5.8e7                      # S/m
angular_speed = 1.0                                # rad/s
field_strength = 0.2                               # T
field_width = 0.02                                 # m, Gaussian standard deviation
magnet_offset = 0.055                              # m, along x
divisions = 32
fibre_inner_radius = 0.2 * radius
fibre_outer_radius = 0.85 * radius
material_radii = np.array([fibre_inner_radius, fibre_outer_radius, radius])
labels = ("Isotropic", "Radial fibres", "Circumferential fibres")
radial_conductivity = reference_conductivity * np.array([1.0, 1.9, 0.1])
circumferential_conductivity = reference_conductivity * np.array([1.0, 0.1, 1.9])

mesh = Mesh.concentric_disk(material_radii, divisions)
radial = mesh.face_centers.normalized()                                     # Vector[F]
directions = stack((radial, mv.xy | radial), axis=-1)                        # [axes] Vector[F]
principal = thickness * np.stack((radial_conductivity, circumferential_conductivity), axis=-1)
# The same mean conductivity fills the fibre-free centre and outer rim.
mean_conductance = thickness * reference_conductivity
radius_at_face = mesh.face_centers.norm()                                  # Scalar[F]
fibre_band = as_scalar((radius_at_face >= fibre_inner_radius) & (radius_at_face <= fibre_outer_radius)).field()
local_principal = mean_conductance + fibre_band * (principal - mean_conductance)  # [cases, axes] Scalar[F]
# Each direction reads an electric-field component and sends current along itself.
# Sum the weighted dyads: a different local material map on every triangle.
conductivity = (directions * (directions | Vector) * local_principal).sum(axis=-1)  # [cases] Conductivity[F]

centre = mv.x * magnet_offset                                              # [] Vector
spin = mv.xy * angular_speed                                                # [] Bivector
# A local patch drives loops; a uniform axial field would be cancelled by a radial potential.
distance_squared = (mesh.edge_midpoints - centre).scalar_norm_squared()    # Scalar[E]
magnetic = mv.xy * field_strength * (-distance_squared / (2 * field_width**2)).exp()  # Bivector[E]


def solve(mesh: Mesh, magnetic: Bivector, spin: Bivector,
          conductivity: Conductivity) -> core.Response:
    velocity = -(spin | mesh.edge_midpoints)                               # Vector[E]
    lorentz = magnetic | Vector                                            # Vector[E] <- Vector
    emf = lorentz(velocity) | mesh.edge_vectors                             # Scalar[E]

    # The DEC derivative maps vertex potentials to edge voltages. Opening Scalar
    # makes its couplings unary maps, ready to compose with the reconstruction.
    derivative = mesh.d0 * Scalar                                         # [E] Scalar <- [V] Scalar
    # Whitney reconstruction evaluates edge circulations as face-centroid vectors.
    # Applied after the derivative, it gives the gradient of a linear face potential.
    reconstruction = mesh.reconstruction                                   # [F] Vector <- [E] Scalar
    gradient = reconstruction(derivative)                                  # [F] Vector <- [V] Scalar
    drive = reconstruction(emf)                                            # Vector[F]
    # Conductivity acts on vectors. Areas integrate their work; the adjoints of
    # reconstruction and derivative assemble it back into vertex charge balance.
    material = spdiag(mesh.triangle_areas * conductivity)                   # [F] Vector <- [F] Vector
    balance = gradient.adjoint()(material(gradient))                       # [V] Scalar <- [V] Scalar
    # Fix one potential to choose the gauge, using its own diagonal for the scale.
    reference = as_scalar(np.arange(len(mesh.vertices.batch())) == 0).field()   # Scalar[V]
    gauge = spdiag(balance.diagonal() * reference)                         # [V] Scalar <- [V] Scalar
    potential = (balance + gauge).solve(gradient.adjoint()(material(drive)))  # Scalar[V]
    voltage = emf - derivative(potential)                                  # Scalar[E]
    electric = reconstruction(voltage)                                     # Vector[F]
    current = conductivity(electric)                                       # Vector[F]
    # The reconstructed currents return to edge fluxes through the adjoint.
    # Their oriented vertex sum vanishes, including at the insulating rim.
    flux = reconstruction.adjoint()(mesh.triangle_areas * current)          # Scalar[E]
    force = flux * lorentz(mesh.edge_vectors)                               # Vector[E]
    torque = (mesh.edge_midpoints ^ force).batch().sum(axis=-1)             # [] Bivector
    heating = mesh.triangle_areas * (electric | current)                    # Scalar[F]
    return core.Response(potential, voltage, flux, current, heating, torque)


response = solve(mesh, magnetic, spin, conductivity)
render.draw(mesh, response, centre, field_width, labels);

The arrows follow the solved sheet currents, with a shared nonlinear colour scale that makes weak currents visible; the dashed circle marks the magnetic patch's width. Both fibre patterns conduct nineteen times more readily along their fibres than across them, but keep the isotropic disc's mean conductivity. The conductivity acts on the field before the adjoint assembles the balance: neighbouring edge contributions are coupled through each face's material map.

In [ ]:
# The torque opposes the positive xy spin; its lost power heats the disc.
print(render.readout(response, labels))

Release a disc with straight fibres and let magnetic drag slow it down. The fibres turn with the material while the magnet stays fixed, so the conductivity extensor changes orientation and the current paths reshape. We advance the rotor and angular momentum in time, solving the electric current at each orientation. The electrical response is quasi-static: this does not model the build-up of induced magnetic fields.

In [ ]:
inertia = 0.005                                    # kg m², disc and attached rotor
initial_speed = 2.0                               # rad/s
initial_angle = 0.0                               # rad, fibres initially along x
along_conductivity = reference_conductivity * 1.9
across_conductivity = reference_conductivity * 0.1
elapsed = 12.0                                    # s
steps = 120
step = elapsed / steps
duration_ms = 80
fibre_lines = 12
fibre_samples = 64
straight_labels = ("Straight fibres",)

body_axes = stack((mv.x, mv.y))                                           # [axes] Vector
body_principal = thickness * np.array([along_conductivity, across_conductivity])
body_conductivity = (body_axes * (body_axes | Vector) * body_principal).sum(axis=-1)[None]  # [cases] Conductivity


def braking() -> Iterator[core.Motion]:
    orientation = (mv.xy * (-initial_angle / 2)).exp()[None]               # [cases] Rotor
    spin = (mv.xy * initial_speed)[None]                                    # [cases] Bivector
    heat = mv.scalar([0]).broadcast_to(spin.shape)                          # [cases] Scalar
    # Pull the input into the body frame, apply its material law, turn the current back.
    material = orientation >> body_conductivity(orientation << Vector)     # [cases] Conductivity
    response = solve(mesh, magnetic, spin, material)
    yield core.Motion(orientation, spin, heat, response)

    for _ in range(steps):
        # Predict the fibre orientation halfway through the step. At fixed orientation
        # currents and torque scale with spin, so one unit-spin solve gives the drag.
        midpoint_orientation = (spin * (-step / 4)).exp() * orientation     # [cases] Rotor
        material = midpoint_orientation >> body_conductivity(midpoint_orientation << Vector)  # [cases] Conductivity
        unit_response = solve(mesh, magnetic, mv.xy, material)
        drag = mv.xy | unit_response.torque                               # [cases] Scalar
        # Use the mean spin for the drag. This midpoint step dissipates exactly the
        # rotational energy counted by its integrated Joule heating.
        midpoint_spin = spin / (1 + drag * step / (2 * inertia))            # [cases] Bivector
        orientation = (midpoint_spin * (-step / 2)).exp() * orientation     # [cases] Rotor
        spin = 2 * midpoint_spin - spin                                   # [cases] Bivector
        heat = heat + step * midpoint_spin.scalar_norm_squared() * unit_response.heating.batch().sum(axis=-1)
        material = orientation >> body_conductivity(orientation << Vector)  # [cases] Conductivity
        response = solve(mesh, magnetic, spin, material)
        yield core.Motion(orientation, spin, heat, response)


straight_fibres = (render.parallel_fibres(radius, fibre_lines, fibre_samples),)
render.inline(render.animate(mesh, braking(), centre, field_width, straight_labels, straight_fibres), duration_ms)

Release the isotropic, radial-fibre and circumferential-fibre discs with the same inertia and initial speed. Their material patterns are unchanged by rotation, so each solved current pattern keeps its shape while its magnitude follows the slowing spin. One unit-spin field solve per material supplies the constant drag; the rim markers show rotation. The nonlinear colour scale stays fixed across panels and time, and the current arrows fade gradually as the discs slow. The isotropic disc brakes hardest, while the two fibre patterns keep turning longer.

In [ ]:
# The spatial material maps stay fixed in the laboratory under rotation.
unit_response = solve(mesh, magnetic, mv.xy, conductivity)
drag = mv.xy | unit_response.torque                                       # [cases] Scalar


def materials_braking() -> Iterator[core.Motion]:
    spin = (mv.xy * initial_speed).broadcast_to(drag.shape)                # [cases] Bivector
    orientation = mv.rotor().broadcast_to(drag.shape)                      # [cases] Rotor
    heat = mv.scalar([0]).broadcast_to(drag.shape)                         # [cases] Scalar
    # Potential, current and torque scale with spin; Joule heating scales with its square.
    response = unit_response.scaled(-(mv.xy | spin))
    yield core.Motion(orientation, spin, heat, response)

    for _ in range(steps):
        # The same midpoint mechanical step, using each material's constant drag.
        midpoint_spin = spin / (1 + drag * step / (2 * inertia))           # [cases] Bivector
        orientation = (midpoint_spin * (-step / 2)).exp() * orientation    # [cases] Rotor
        spin = 2 * midpoint_spin - spin                                  # [cases] Bivector
        heat = heat + step * midpoint_spin.scalar_norm_squared() * unit_response.heating.batch().sum(axis=-1)
        response = unit_response.scaled(-(mv.xy | spin))
        yield core.Motion(orientation, spin, heat, response)


radial_fibres = render.radial_fibres(fibre_inner_radius, fibre_outer_radius, fibre_lines, fibre_samples)
circular_fibres = render.circular_fibres(fibre_inner_radius, fibre_outer_radius, fibre_lines, fibre_samples)
# The isotropic material has no preferred fibre direction.
fibres = (radial_fibres[:0], radial_fibres, circular_fibres)
render.inline(render.animate(mesh, materials_braking(), centre, field_width, labels, fibres), duration_ms)

Heat stays with the rotating material. Each triangle keeps the Joule energy it has received as it travels beneath the fixed magnet. We sample the magnetic field on the moving edges and pull it back into the material frame, so the same sparse operators solve on the rotating domain. The colours show accumulated energy per area on one fixed scale; thermal diffusion and cooling are omitted. The electrical response remains instantaneous, but the heat distribution remembers where the material has been.

In [ ]:
def material_heating() -> Iterator[core.ThermalMotion]:
    orientation = mv.rotor().broadcast_to(conductivity.shape)              # [cases] Rotor
    spin = (mv.xy * initial_speed).broadcast_to(orientation.shape)         # [cases] Bivector
    heat = (mesh.triangle_areas * 0).broadcast_to(conductivity.shape)      # [cases] Scalar[F]
    vertices = orientation >> mesh.vertices                               # [cases] Vector[V]
    yield core.ThermalMotion(orientation, spin, vertices, heat)

    for _ in range(steps):
        midpoint_orientation = (spin * (-step / 4)).exp() * orientation    # [cases] Rotor
        # The mesh moves through the fixed magnetic patch. Pull the sampled field
        # into the material frame, where geometry and conductivity keep their labels.
        positions = midpoint_orientation >> mesh.edge_midpoints           # [cases] Vector[E]
        distance_squared = (positions - centre).scalar_norm_squared()     # [cases] Scalar[E]
        magnetic = mv.xy * field_strength * (-distance_squared / (2 * field_width**2)).exp()  # [cases] Bivector[E]
        body_magnetic = midpoint_orientation << magnetic                  # [cases] Bivector[E]
        body_plane = midpoint_orientation << mv.xy                       # [cases] Bivector
        unit_response = solve(mesh, body_magnetic, body_plane, conductivity)
        drag = body_plane | unit_response.torque                         # [cases] Scalar
        midpoint_spin = spin / (1 + drag * step / (2 * inertia))           # [cases] Bivector

        # Deposit energy into each material face, using the spin that sets the drag.
        # The sum of these deposits equals the lost rotational kinetic energy.
        heat = heat + step * midpoint_spin.scalar_norm_squared() * unit_response.heating
        orientation = (midpoint_spin * (-step / 2)).exp() * orientation    # [cases] Rotor
        spin = 2 * midpoint_spin - spin                                  # [cases] Bivector
        # The heat stays on its triangle as that triangle moves: no interpolation.
        vertices = orientation >> mesh.vertices                           # [cases] Vector[V]
        yield core.ThermalMotion(orientation, spin, vertices, heat)


render.inline(render.animate_heat(mesh, material_heating(), centre, field_width, labels, fibres), duration_ms)


Current takes time to build because its magnetic field stores energy. A brief motor pulse accelerates the discs from rest, then releases them to magnetic braking. We evolve closed current patterns and rotation using a drive projected over one revolution. Modal damping accounts for total heat; weighted amplitude histories locate that heat at display frames without losing the contributions between modes. The motor’s work supplies the combined rotational energy, magnetic energy and heat.

In [ ]:
permeability = 4 * np.pi * 1e-7                    # H/m, free space
inductive_divisions = divisions
mode_count = 63
drive_harmonics = 16
inductive_field_strength = 1.0                     # T
inductive_initial_speed = 0.0                      # rad/s
inductive_target_speed = 600.0                     # rad/s, unloaded motor pulse
spinup_time = 0.012                                # s
inductive_elapsed = 0.1                            # s
inductive_steps = 120
inductive_step = inductive_elapsed / inductive_steps
electrical_substeps = 8

inductive_mesh = Mesh.concentric_disk(material_radii, inductive_divisions)
local_radial = inductive_mesh.face_centers.normalized()                    # Vector[F]
local_axes = stack((local_radial, mv.xy | local_radial), axis=-1)            # [axes] Vector[F]
radius_at_face = inductive_mesh.face_centers.norm()                         # Scalar[F]
fibre_band = as_scalar((radius_at_face >= fibre_inner_radius) & (radius_at_face <= fibre_outer_radius)).field()
local_principal = mean_conductance + fibre_band * (principal - mean_conductance)  # [cases, axes] Scalar[F]
inductive_conductivity = (local_axes * (local_axes | Vector) * local_principal).sum(axis=-1)  # [cases] Conductivity[F]

# A streamfunction fixed to zero on the rim makes the current tangent to that rim.
# Its turned face gradient has matching normal flux on both sides of every interior edge.
boundary = np.unique(inductive_mesh.edges[inductive_mesh.boundary_edges])
vertex_count = len(inductive_mesh.vertices.batch())
interior = np.setdiff1d(np.arange(vertex_count), boundary)
embedder = SparseExtensor.from_indices(mv.scalar(np.ones((len(interior), 1))),
                                       interior, np.arange(len(interior)),
                                       (vertex_count, len(interior))) * Scalar
current_gradient = inductive_mesh.reconstruction((inductive_mesh.d0 * Scalar)(embedder))  # [F] Vector <- [interior] Scalar
current = spdiag(inductive_mesh.face_planes | Vector)(current_gradient)     # [F] Vector <- [interior] Scalar
# Each face's map is inverted on its own, as a batch of independent maps.
resistivity = inductive_conductivity.batch().lstsq(Vector).field()          # [cases] Conductivity[F]
resistance = current.adjoint()(spdiag(inductive_mesh.triangle_areas * resistivity)(current))  # [cases, interior] Scalar <- [interior] Scalar
# Analytic triangle integrals couple every face current to every other face current.
# Pulling this map through the current map and its adjoint gives the inductance.
couplings = core.magnetic_coupling_blocks(inductive_mesh, permeability, core.MAGNETIC_BLOCK_FACES)
current_adjoint = current.adjoint()
inductance = current_adjoint(next(couplings)(current))                    # [interior] Scalar <- [interior] Scalar
for coupling in couplings:
    inductance = inductance + current_adjoint(coupling(current))
inductance = (inductance + inductance.adjoint()) * 0.5
decay, eigenfields = resistance.eigh(inductance, mode_count)                # [cases, modes] Scalar; [cases, modes] Scalar[interior]
# Each pattern has unit inductance: squared amplitudes measure twice the magnetic energy.
modes = core.CurrentModes(decay, current(eigenfields), embedder(eigenfields), resistivity)


def periodic_drive(mesh: Mesh, modes: core.CurrentModes, centre: Vector, plane: Bivector,
                   strength: float, width: float, orientation: Rotor, count: int) -> core.PeriodicDrive:
    samples = 2 * count + 1
    angles = np.linspace(0, 2 * np.pi, samples, endpoint=False)
    turns = (plane * (-angles / 2)).exp()
    orientations = turns[:, None] * orientation                          # [samples, cases] Rotor
    positions = orientations >> mesh.face_centers                        # [samples, cases] Vector[F]
    magnetic = orientations << core.field(positions, centre, plane, strength, width)
    body_plane = orientation << plane                                   # [cases] Bivector
    velocity = -(body_plane | mesh.face_centers)                          # [cases] Vector[F]
    electric = magnetic | velocity                                      # [samples, cases] Vector[F]
    # Project one angle at a time, avoiding a samples × modes × faces temporary.
    values = stack([(modes.currents | (mesh.triangle_areas * force)[..., None]).batch().sum(axis=-1)
                    for force in electric], axis=-2)                    # [cases, samples, modes] Scalar
    harmonics = np.arange(1, count + 1)
    phase = harmonics[:, None] * angles
    cosine = (values[..., None, :, :] * (2 / samples * np.cos(phase))[..., None]).sum(axis=-2)
    sine = (values[..., None, :, :] * (2 / samples * np.sin(phase))[..., None]).sum(axis=-2)
    return core.PeriodicDrive(values.mean(axis=-2), cosine, sine, harmonics)



def inductive_braking(mesh: Mesh, modes: core.CurrentModes, forcing: core.PeriodicDrive, plane: Bivector,
                      inertia: float,
                      orientation: Rotor, spin: Bivector, heat: Scalar, amplitudes: Scalar,
                      motor_torque: Bivector, step: float) -> Iterator[core.InductiveMotion]:
    substeps = motor_torque.shape[1]
    interval = step / substeps
    angle = mv.scalar([0]).broadcast_to(orientation.shape)                # [cases] Scalar
    thermal_energy = heat.batch().sum(axis=-1)                           # [cases] Scalar
    motor_work = thermal_energy * 0                                      # [cases] Scalar
    factors = mv.scalar(np.zeros((substeps,) + amplitudes.shape + (1,)))  # [substeps, cases, modes] Scalar
    damping = 1 + interval * modes.decay / 2                              # [cases, modes] Scalar
    yield modes.state(mesh, orientation, spin, heat, amplitudes, thermal_energy, motor_work)
    for torques in motor_torque:
        for substep, torque in enumerate(torques):
            speed = -(plane | spin)                                           # [cases] Scalar
            motor = -(plane | torque)                                         # [cases] Scalar
            drive = forcing(angle + speed * (interval / 2))                   # [cases, modes] Scalar
            free_amplitudes = amplitudes / damping                            # [cases, modes] Scalar
            driven_amplitudes = interval / 2 * drive / damping                 # [cases, modes] Scalar
            # Eliminate the midpoint currents from angular momentum, then recover
            # both states together. This includes energy returning from the magnetic field.
            midpoint_speed = (speed + interval / (2 * inertia) * (motor - (drive * free_amplitudes).sum(axis=-1))) / (
                1 + interval / (2 * inertia) * (drive * driven_amplitudes).sum(axis=-1))  # [cases] Scalar
            midpoint_amplitudes = free_amplitudes + midpoint_speed[..., None] * driven_amplitudes  # [cases, modes] Scalar
            # Modal damping gives total heat exactly. Keep weighted amplitudes to
            # reconstruct its location, including cross terms, at the display frame.
            factors = factors.at[substep].set(np.sqrt(interval) * midpoint_amplitudes)
            thermal_energy = thermal_energy + interval * (modes.decay * midpoint_amplitudes.squared()).sum(axis=-1)
            motor_work = motor_work + interval * motor * midpoint_speed
            amplitudes = 2 * midpoint_amplitudes - amplitudes                   # [cases, modes] Scalar
            midpoint_spin = plane * midpoint_speed                            # [cases] Bivector
            angle = angle + interval * midpoint_speed                         # [cases] Scalar
            orientation = (midpoint_spin * (-interval / 2)).exp() * orientation  # [cases] Rotor
            spin = 2 * midpoint_spin - spin                                    # [cases] Bivector
        heat = heat + modes.deposited_heat(mesh, factors)                     # [cases] Scalar[F]
        yield modes.state(mesh, orientation, spin, heat, amplitudes, thermal_energy, motor_work)



orientation = mv.rotor().broadcast_to(inductive_conductivity.shape)        # [cases] Rotor
spin = (mv.xy * inductive_initial_speed).broadcast_to(orientation.shape)    # [cases] Bivector
heat = (inductive_mesh.triangle_areas * 0).broadcast_to(inductive_conductivity.shape)  # [cases] Scalar[F]
amplitudes = mv.scalar([0]).broadcast_to(decay.shape)                      # [cases, modes] Scalar
forcing = periodic_drive(inductive_mesh, modes, centre, mv.xy, inductive_field_strength,
                         field_width, orientation, drive_harmonics)
# A short motor pulse accelerates the disc; its work supplies motion, field energy and heat.
times = (np.arange(inductive_steps * electrical_substeps) + 0.5) * inductive_step / electrical_substeps
phase = np.clip(times / spinup_time, 0, 1)
acceleration = inductive_target_speed * np.pi / (2 * spinup_time) * np.sin(np.pi * phase) * (times < spinup_time)
motor_torque = mv.xy * (inertia * acceleration.reshape(inductive_steps, electrical_substeps, 1))
inductive_states = list(inductive_braking(inductive_mesh, modes, forcing, mv.xy, inertia,
                                        orientation, spin, heat, amplitudes,
                                        motor_torque, inductive_step))
render.inline(render.animate_inductive(inductive_mesh, inductive_states, centre, field_width, labels, fibres), duration_ms)


In [ ]:
# The same current history heats the material faces as they rotate past the magnet.
render.inline(render.animate_heat(inductive_mesh, inductive_states, centre, field_width, labels, fibres), duration_ms)


The disc is a thin conducting sheet with illustrative directional conductivities. DEC edge differences and Whitney reconstruction give face gradients. The potential solves enforce charge balance weakly; the streamfunction currents have continuous normal flux and an insulating rim by construction. Colours interpolate face values at vertices for display. Potential-solve streamlines interpolate face currents for display. Inductive current paths use fixed levels of the solved streamfunction, with light corner smoothing for display. The inductive model includes free-space magnetic coupling with analytic source-triangle integrals and a finite set of current patterns; faster omitted patterns, mesh resolution and time resolution set its accuracy. The magnet stays prescribed; displacement current, variation through the thickness, thermal diffusion and cooling are neglected. [Discrete 2-Tensor Fields on Triangulations](https://geometry.caltech.edu/pubs/GLBTD14.pdf) develops material-dependent operators in discrete exterior calculus. [Magnetic-field modeling with surface currents](https://arxiv.org/abs/2005.10060) develops the streamfunction and triangle-integral construction, and the [COMSOL magnetic-brake model](https://doc.comsol.com/6.4/doc/com.comsol.help.models.acdc.magnetic_brake/magnetic_brake.html) treats a volume conductor.